In [2]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

# Load data
df = pd.read_csv('cleaned_zomato.csv')

# Clean text
df['City'] = df['City'].astype(str).str.strip()
df['Locality'] = df['Locality'].astype(str).str.strip()

# Aggregate
market_data = df.groupby(['City', 'Locality']).agg({
    'rating': 'mean',
    'votes': 'sum',
    'cost': 'mean',
    'Restaurant Name': 'count'
}).rename(columns={'Restaurant Name': 'Total_Restaurants'}).reset_index()

# Feature Engineering
market_data['votes_log'] = np.log1p(market_data['votes'])
market_data['votes_per_restaurant'] = market_data['votes'] / market_data['Total_Restaurants']
market_data['cost_per_vote'] = market_data['cost'] / (market_data['votes'] + 1)

# Features
features = ['rating', 'votes_log', 'votes_per_restaurant', 'cost', 'cost_per_vote']

# Scaling
scaler = StandardScaler()
scaled_features = scaler.fit_transform(market_data[features])

# Model
kmeans = KMeans(n_clusters=4, random_state=42)
market_data['Cluster_ID'] = kmeans.fit_predict(scaled_features)

# Evaluation
score = silhouette_score(scaled_features, market_data['Cluster_ID'])
print("Silhouette Score:", score)

# Cluster centers
centers = scaler.inverse_transform(kmeans.cluster_centers_)
cluster_info = pd.DataFrame(centers, columns=features)

# Persona mapping
def assign_persona(idx):
    row = cluster_info.iloc[idx]

    if row['votes_per_restaurant'] > cluster_info['votes_per_restaurant'].mean():
        return "High Demand Hotspot 🔥"
    elif row['rating'] > cluster_info['rating'].mean():
        return "Premium Quality Zone 💎"
    elif row['cost_per_vote'] > cluster_info['cost_per_vote'].mean():
        return "Overpriced / Low Efficiency ❌"
    else:
        return "Emerging / Low Visibility ⚠️"

market_data['Strategic_Persona'] = market_data['Cluster_ID'].apply(assign_persona)

# Save
market_data.to_csv("market_data.csv", index=False)

print("✅ market_data.csv saved successfully")

Silhouette Score: 0.2997918939442979
✅ market_data.csv saved successfully
